In [1]:
# ================================================
# Chest X-ray Pneumonia Classification
# Notebook 5: Final Comparison & W&B Tracking
# ================================================

In [3]:
import subprocess
subprocess.run(["pip", "uninstall", "wandb", "-y"])
subprocess.run(["pip", "install", "wandb==0.16.6"])
print("Done! Now restart kernel again.")

Done! Now restart kernel again.


In [4]:
import os
os.environ["WANDB_API_KEY"] = "wandb_v1_5sEsGHOTX5UeuIMZxxWyQAqyILV"
os.environ["WANDB_SILENT"] = "true"

import wandb
print("wandb version:", wandb.__version__)
print("W&B ready!")

wandb version: 0.27.2
W&B ready!


In [5]:
import numpy as np
import matplotlib.pyplot as plt
import os
import wandb

print("All imports ready!")
print("wandb version:", wandb.__version__)

All imports ready!
wandb version: 0.27.2


In [7]:
import os
import wandb

os.environ["WANDB_API_KEY"] = "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"
os.environ["WANDB_SILENT"] = "true"

# Test connection
run = wandb.init(project="chest-xray-pneumonia", name="connection-test")
wandb.finish()

print("W&B connected successfully!")

WandbCoreNotAvailableError: File not found: c:\Users\maria\Documents\chest_xray_project\venv\Lib\site-packages\wandb\bin\wandb-core. Please contact support at support@wandb.com. Your platform is: Windows-11-10.0.26200-SP0.

In [10]:
import wandb
print("wandb version:", wandb.__version__)

wandb version: 0.27.2


In [5]:
import wandb
import os

# Directly set your API key - paste it between the quotes
os.environ["WANDB_API_KEY"] = "wandb_v1_5sEsGHOTX5UeuIMZxxWyQAqyILV"

# Test connection
wandb.init(project="chest_xray", name="connection-test")
wandb.finish()

print("W&B connected successfully!")

CommError: user is not logged in

In [ ]:
# ================================================
# Log all model results to W&B
# ================================================

# Our results collected from all notebooks
results = {
    "ResNet18_CNN": {
        "test_accuracy":      0.85,
        "normal_precision":   0.97,
        "normal_recall":      0.63,
        "normal_f1":          0.76,
        "pneumonia_precision":0.82,
        "pneumonia_recall":   0.99,
        "pneumonia_f1":       0.89,
        "macro_f1":           0.83,
        "trainable_params":   1026,
        "total_params":       11177538,
        "epochs":             5,
    },
    "ViT_Frozen": {
        "test_accuracy":      0.86,
        "normal_precision":   0.96,
        "normal_recall":      0.65,
        "normal_f1":          0.77,
        "pneumonia_precision":0.82,
        "pneumonia_recall":   0.98,
        "pneumonia_f1":       0.89,
        "macro_f1":           0.83,
        "trainable_params":   1538,
        "total_params":       85800194,
        "epochs":             5,
    },
    "ViT_Full_Finetune": {
        "test_accuracy":      0.86,
        "normal_precision":   0.99,
        "normal_recall":      0.62,
        "normal_f1":          0.76,
        "pneumonia_precision":0.81,
        "pneumonia_recall":   1.00,
        "pneumonia_f1":       0.90,
        "macro_f1":           0.83,
        "trainable_params":   85800194,
        "total_params":       85800194,
        "epochs":             3,
    },
    "LoRA_ViT": {
        "test_accuracy":      0.90,
        "normal_precision":   0.98,
        "normal_recall":      0.74,
        "normal_f1":          0.84,
        "pneumonia_precision":0.86,
        "pneumonia_recall":   0.99,
        "pneumonia_f1":       0.92,
        "macro_f1":           0.88,
        "trainable_params":   296450,
        "total_params":       86096644,
        "epochs":             5,
    }
}

# Log each model as a separate W&B run
for model_name, metrics in results.items():
    run = wandb.init(
        project="chest-xray-pneumonia",
        name=model_name,
        config={"model": model_name, "epochs": metrics["epochs"]}
    )
    wandb.log(metrics)
    wandb.finish()
    print(f"Logged {model_name} to W&B ✓")

print("\nAll models logged to W&B!")
print("Go to wandb.ai to see your dashboard")

In [ ]:
# ================================================
# Plot 1: Accuracy Comparison
# ================================================

models     = ["ResNet18\n(CNN)", "ViT\n(Frozen)", "ViT\n(Full)", "LoRA\n(ViT)"]
accuracies = [85, 86, 86, 90]
colors     = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"]

plt.figure(figsize=(10, 6))
bars = plt.bar(models, accuracies, color=colors, width=0.5, edgecolor="black")

# Add value labels on bars
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.3,
             f"{acc}%", ha="center", va="bottom", fontweight="bold", fontsize=12)

plt.ylim(80, 95)
plt.title("Test Accuracy Comparison Across Models", fontsize=14, fontweight="bold")
plt.ylabel("Test Accuracy (%)", fontsize=12)
plt.xlabel("Model", fontsize=12)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("../outputs/accuracy_comparison.png", dpi=150)
plt.show()
print("Saved!")

In [ ]:
# ================================================
# Plot 2: Precision & Recall per Class
# ================================================

models_list = ["ResNet18", "ViT Frozen", "ViT Full", "LoRA ViT"]
x           = np.arange(len(models_list))
width       = 0.2

normal_precision    = [0.97, 0.96, 0.99, 0.98]
normal_recall       = [0.63, 0.65, 0.62, 0.74]
pneumonia_precision = [0.82, 0.82, 0.81, 0.86]
pneumonia_recall    = [0.99, 0.98, 1.00, 0.99]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# NORMAL class
ax1.bar(x - width/2, normal_precision, width, label="Precision", color="#4C72B0")
ax1.bar(x + width/2, normal_recall,    width, label="Recall",    color="#55A868")
ax1.set_title("NORMAL Class — Precision vs Recall", fontsize=12, fontweight="bold")
ax1.set_xticks(x)
ax1.set_xticklabels(models_list)
ax1.set_ylim(0, 1.1)
ax1.set_ylabel("Score")
ax1.legend()
ax1.grid(axis="y", alpha=0.3)

for i, (p, r) in enumerate(zip(normal_precision, normal_recall)):
    ax1.text(i - width/2, p + 0.02, f"{p:.2f}", ha="center", fontsize=9)
    ax1.text(i + width/2, r + 0.02, f"{r:.2f}", ha="center", fontsize=9)

# PNEUMONIA class
ax2.bar(x - width/2, pneumonia_precision, width, label="Precision", color="#C44E52")
ax2.bar(x + width/2, pneumonia_recall,    width, label="Recall",    color="#8172B2")
ax2.set_title("PNEUMONIA Class — Precision vs Recall", fontsize=12, fontweight="bold")
ax2.set_xticks(x)
ax2.set_xticklabels(models_list)
ax2.set_ylim(0, 1.1)
ax2.set_ylabel("Score")
ax2.legend()
ax2.grid(axis="y", alpha=0.3)

for i, (p, r) in enumerate(zip(pneumonia_precision, pneumonia_recall)):
    ax2.text(i - width/2, p + 0.02, f"{p:.2f}", ha="center", fontsize=9)
    ax2.text(i + width/2, r + 0.02, f"{r:.2f}", ha="center", fontsize=9)

plt.suptitle("Precision & Recall Comparison Across Models", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/precision_recall_comparison.png", dpi=150)
plt.show()
print("Saved!")

In [ ]:
# ================================================
# Plot 3: Parameter Efficiency
# Most important plot for your report!
# ================================================

models_list    = ["ResNet18", "ViT Frozen", "ViT Full", "LoRA ViT"]
trainable_pct  = [0.009, 0.002, 100.0, 0.344]  # % of params trained
test_accuracies= [85, 86, 86, 90]
colors         = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"]
sizes          = [200, 200, 800, 400]  # bubble size proportional to total params

fig, ax = plt.subplots(figsize=(10, 7))

scatter = ax.scatter(trainable_pct, test_accuracies,
                     c=colors, s=sizes, alpha=0.8, edgecolors="black")

# Labels for each point
offsets = [(-2, 0.3), (-2, 0.3), (1, 0.3), (0.1, 0.3)]
for i, (model, x, y) in enumerate(zip(models_list, trainable_pct, test_accuracies)):
    ax.annotate(model,
                xy=(x, y),
                xytext=(x + offsets[i][0], y + offsets[i][1]),
                fontsize=11, fontweight="bold")

ax.set_xlabel("Trainable Parameters (%)", fontsize=12)
ax.set_ylabel("Test Accuracy (%)", fontsize=12)
ax.set_title("Parameter Efficiency vs Accuracy\n(Bubble size = total parameters)",
             fontsize=13, fontweight="bold")
ax.set_ylim(83, 93)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("../outputs/parameter_efficiency.png", dpi=150)
plt.show()
print("Saved!")

In [ ]:
# ================================================
# Final Summary Table
# ================================================

fig, ax = plt.subplots(figsize=(13, 4))
ax.axis("off")

table_data = [
    ["ResNet18 (CNN)",    "1,026",      "11,177,538",  "0.009%", "85%", "0.76", "0.89", "0.83"],
    ["ViT Frozen",        "1,538",      "85,800,194",  "0.002%", "86%", "0.77", "0.89", "0.83"],
    ["ViT Full Fine-tune","85,800,194", "85,800,194",  "100%",   "86%", "0.76", "0.90", "0.83"],
    ["LoRA ViT ⭐",       "296,450",    "86,096,644",  "0.344%", "90%", "0.84", "0.92", "0.88"],
]

columns = ["Model", "Trainable\nParams", "Total\nParams",
           "% Trained", "Test\nAcc", "F1\nNORMAL",
           "F1\nPNEUMONIA", "Macro\nF1"]

table = ax.table(
    cellText=table_data,
    colLabels=columns,
    loc="center",
    cellLoc="center"
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2.2)

# Highlight header
for j in range(len(columns)):
    table[0, j].set_facecolor("#2C3E50")
    table[0, j].set_text_props(color="white", fontweight="bold")

# Highlight LoRA row (best model)
for j in range(len(columns)):
    table[4, j].set_facecolor("#D5F5E3")
    table[4, j].set_text_props(fontweight="bold")

plt.title("Final Model Comparison Summary",
          fontsize=14, fontweight="bold", pad=20)
plt.tight_layout()
plt.savefig("../outputs/final_summary_table.png", dpi=150, bbox_inches="tight")
plt.show()
print("Summary table saved!")

In [ ]:
# ================================================
# Final Conclusions for Report
# ================================================

print("=" * 60)
print("FINAL PROJECT CONCLUSIONS")
print("=" * 60)

print("""
1. BEST MODEL: LoRA ViT
   - Highest accuracy: 90%
   - Best NORMAL recall: 74%
   - Best Macro F1: 0.88
   - Only trained 0.344% of parameters

2. KEY FINDING — Parameter Efficiency:
   - LoRA trained 296,450 parameters
   - Full ViT trained 85,800,194 parameters
   - That is 290x MORE parameters for the SAME or WORSE result

3. CLASS IMBALANCE OBSERVATION:
   - All models struggled with NORMAL recall (62-74%)
   - All models excelled at PNEUMONIA recall (98-100%)
   - This is due to 3x more Pneumonia images in training data
   - LoRA handled this best (74% NORMAL recall)

4. RECOMMENDATION:
   - For medical imaging with limited compute resources,
     LoRA fine-tuning of Vision Transformers is the
     most efficient and effective approach

5. LIMITATIONS:
   - Validation set too small (only 16 images)
   - All experiments run on CPU (no GPU available)
   - Future work: larger val set, GPU training,
     higher LoRA rank experimentation
""")